In [0]:
# ----------------------------
# Standard widgets (all layers)
# ----------------------------
dbutils.widgets.text("base_path", "/Volumes/workspace/ecommerce/ecommerce_data")
dbutils.widgets.text("raw_path", "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")  # only used by Bronze
dbutils.widgets.text("bronze_path", "")  # default derived if blank
dbutils.widgets.text("silver_path", "")  # default derived if blank
dbutils.widgets.text("gold_path", "")    # default derived if blank
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"])  # basic control

base_path = dbutils.widgets.get("base_path").strip()
raw_path = dbutils.widgets.get("raw_path").strip()
bronze_path = dbutils.widgets.get("bronze_path").strip()
silver_path = dbutils.widgets.get("silver_path").strip()
gold_path = dbutils.widgets.get("gold_path").strip()
mode = dbutils.widgets.get("mode").strip()

# Derive defaults if not provided
if not bronze_path:
    bronze_path = f"{base_path}/bronze/events_raw"
if not silver_path:
    silver_path = f"{base_path}/silver/events"
if not gold_path:
    gold_path = f"{base_path}/gold/product_performance_daily"

print(f"base_path={base_path}")
print(f"raw_path={raw_path}")
print(f"bronze_path={bronze_path}")
print(f"silver_path={silver_path}")
print(f"gold_path={gold_path}")
print(f"mode={mode}")





from pyspark.sql import functions as F
  
raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(raw_path)
)

bronze = (
    raw
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingestion_date", F.to_date("ingestion_ts"))
)

writer = (bronze.write.format("delta").partitionBy("ingestion_date"))

if mode == "full":
    writer.mode("overwrite").save(bronze_path)
else:
    writer.mode("append").save(bronze_path)
